<a href="https://colab.research.google.com/github/swetha-off/DSA0302-NLP/blob/main/AT3_CODING_CHALLENGE_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PROBLEM 1 — TOKENIZATION WITH A REGULAR EXPRESSION

In [20]:

import re
N = int(input())
pattern = r"[A-Za-z]+(?:['-][A-Za-z]+)*"
for _ in range(N):
    line = input()
    tokens = re.findall(pattern, line)
    if tokens:
        print(len(tokens), *tokens)
    else:
        print(0)


5
Hello! John's cat is well-behaved.
5 Hello John's cat is well-behaved
Machine-learning helps solve real-world problems in 2026.
6 Machine-learning helps solve real-world problems in
4567 !!! @@@
0
Don't stop now... we're learning Python.
6 Don't stop now we're learning Python
blue--green red's state-of-the-art
4 blue green red's state-of-the-art


# PROBLEM 2 — SIMULATING A FINITE STATE AUTOMATON

In [30]:
import sys

def main():
    n, k = map(int, input().split())
    alphabet = input().split()
    start = int(input())

    data = list(map(int, input().split()))
    accepting = set(data[1:])

    t = int(input())
    transitions = {}

    for _ in range(t):
        p, c, r = input().split()
        p, r = int(p), int(r)
        transitions.setdefault((p, c), set()).add(r)

    Q = int(input())

    for _ in range(Q):
        s = input().rstrip("\n")
        if s == "#":
            s = ""

        states = {start}

        for c in s:
            next_states = set()
            for state in states:
                next_states.update(transitions.get((state, c), set()))
            states = next_states

            if not states:
                break

        print("ACCEPT" if states & accepting else "REJECT")

if __name__ == '__main__':
    main()

4 2 
a b 
0
2 2 3
5
0 a 1
0 b 2
1 b 3
2 a 3
3 b 3
6
a
REJECT
b
ACCEPT
ab
ACCEPT
ba
ACCEPT
 bb
REJECT
#
REJECT


# PROBLEM 3 — INFLECTIONAL MORPHOLOGY GENERATOR

In [32]:

def vowel_groups(stem):
    vowels = "aeiou"
    count = 0
    in_vowel = False

    for ch in stem:
        if ch in vowels:
            if not in_vowel:
                count += 1
            in_vowel = True
        else:
            in_vowel = False

    return count


def doublable(stem):
    vowels = "aeiou"

    if len(stem) < 3:
        return False

    if stem[-3] in vowels or stem[-2] not in vowels or stem[-1] in vowels:
        return False

    if stem[-1] in "wxy":
        return False

    return vowel_groups(stem) == 1


def inflect(stem, feature):
    vowels = "aeiou"

    if feature in ("N+PL", "V+3SG"):
        if stem.endswith(("s", "x", "z", "ch", "sh")):
            return stem + "es"
        if len(stem) >= 2 and stem[-1] == "o" and stem[-2] not in vowels:
            return stem + "es"
        if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
            return stem[:-1] + "ies"
        return stem + "s"

    if feature == "V+PROG":
        if len(stem) >= 3 and stem.endswith("e") and stem[-2] not in vowels:
            return stem[:-1] + "ing"
        if doublable(stem):
            return stem + stem[-1] + "ing"
        return stem + "ing"

    suffixes = {
        "V+PAST": "ed",
        "ADJ+COMP": "er",
        "ADJ+SUP": "est"
    }

    suffix = suffixes[feature]

    if stem.endswith("e"):
        return stem + suffix[1:]

    if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
        return stem[:-1] + "i" + suffix

    if doublable(stem):
        return stem + stem[-1] + suffix

    return stem + suffix


L = int(input())

lexicon = {}

for _ in range(L):
    stem, feature, form = input().split()
    lexicon[(stem, feature)] = form

Q = int(input())

for _ in range(Q):
    stem, feature = input().split()

    if (stem, feature) in lexicon:
        print(lexicon[(stem, feature)])
    else:
        print(inflect(stem, feature))


3
man N+PL men
run V+PAST ran
far ADJ+COMP farther
8
book N+PL
books
box N+PL
boxes
try N+PL
tries
dance V+PROG
dancing
stop V+PAST
stopped
large ADJ+COMP
larger
man N+PL
men
far ADJ+SUP
farrest


# PROBLEM 4 — FINITE-STATE MORPHOLOGICAL PARSING

In [34]:

def vowel_groups(stem):
    vowels = "aeiou"
    count = 0
    in_vowel = False

    for ch in stem:
        if ch in vowels:
            if not in_vowel:
                count += 1
            in_vowel = True
        else:
            in_vowel = False

    return count


def doublable(stem):
    vowels = "aeiou"

    if len(stem) < 3:
        return False

    if not (stem[-3] not in vowels and
            stem[-2] in vowels and
            stem[-1] not in vowels):
        return False

    if stem[-1] in "wxy":
        return False

    return vowel_groups(stem) == 1


def generate(stem, feature):
    vowels = "aeiou"

    if feature in ("PL", "3SG"):
        if stem.endswith(("s", "x", "z", "ch", "sh")):
            return stem + "es"
        if len(stem) >= 2 and stem.endswith("o") and stem[-2] not in vowels:
            return stem + "es"
        if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
            return stem[:-1] + "ies"
        return stem + "s"

    if feature == "PROG":
        if len(stem) >= 3 and stem.endswith("e") and stem[-2] not in vowels:
            return stem[:-1] + "ing"
        if doublable(stem):
            return stem + stem[-1] + "ing"
        return stem + "ing"

    suffixes = {
        "PAST": "ed",
        "COMP": "er",
        "SUP": "est"
    }

    suffix = suffixes[feature]

    if stem.endswith("e"):
        return stem + suffix[1:]

    if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
        return stem[:-1] + "i" + suffix

    if doublable(stem):
        return stem + stem[-1] + suffix

    return stem + suffix


S = int(input())

lexicon = []

for _ in range(S):
    stem, pos = input().split()
    lexicon.append((stem, pos))

Q = int(input())

for _ in range(Q):
    surface = input().strip()
    analyses = []

    for stem, pos in lexicon:
        if surface == stem:
            analyses.append(stem + "+" + pos)

        if pos == "N":
            features = ["PL"]
        elif pos == "V":
            features = ["3SG", "PROG", "PAST"]
        else:
            features = ["COMP", "SUP"]

        for feature in features:
            if generate(stem, feature) == surface:
                analyses.append(stem + "+" + pos + "+" + feature)

    if analyses:
        print(", ".join(sorted(set(analyses))))
    else:
        print("UNKNOWN")


6
book N
box N
play V
dance V
small ADJ
Fast ADJ
8
books
book+N+PL
boxes
box+N+PL
plays
play+V+3SG
dancing
dance+V+PROG
danced
dance+V+PAST
smaller
small+ADJ+COMP
fastest
UNKNOWN
computer
UNKNOWN


# PROBLEM 5 — THE PORTER STEMMER, STEP 1

In [35]:

def is_consonant(word, i):
    ch = word[i]

    if ch in "aeiou":
        return False

    if ch == 'y':
        if i == 0:
            return True
        return not is_consonant(word, i - 1)

    return True


def measure(word):
    m = 0
    i = 0
    n = len(word)

    while i < n and is_consonant(word, i):
        i += 1

    while i < n:
        while i < n and not is_consonant(word, i):
            i += 1

        if i < n:
            m += 1
            while i < n and is_consonant(word, i):
                i += 1

    return m

def contains_vowel(word):
    for i in range(len(word)):
        if not is_consonant(word, i):
            return True
    return False

def ends_double_consonant(word):
    return (
        len(word) >= 2
        and word[-1] == word[-2]
        and is_consonant(word, len(word) - 1)
    )

def cvc(word):
    if len(word) < 3:
        return False

    n = len(word)

    return (
        is_consonant(word, n - 3)
        and not is_consonant(word, n - 2)
        and is_consonant(word, n - 1)
        and word[-1] not in "wxy"
    )

def step1a(word):
    if word.endswith("sses"):
        return word[:-2]
    elif word.endswith("ies"):
        return word[:-2]
    elif word.endswith("ss"):
        return word
    elif word.endswith("s"):
        return word[:-1]

    return word

def step1b(word):
    if word.endswith("eed"):
        stem = word[:-3]

        if measure(stem) > 0:
            return stem + "ee"

        return word

    if word.endswith("ed") and contains_vowel(word[:-2]):
        stem = word[:-2]

    elif word.endswith("ing") and contains_vowel(word[:-3]):
        stem = word[:-3]

    else:
        return word

    if stem.endswith(("at", "bl", "iz")):
        return stem + "e"

    if ends_double_consonant(stem) and stem[-1] not in "lsz":
        return stem[:-1]

    if measure(stem) == 1 and cvc(stem):
        return stem + "e"

    return stem


def step1c(word):
    if word.endswith("y") and contains_vowel(word[:-1]):
        return word[:-1] + "i"

    return word


Q = int(input())

for _ in range(Q):
    word = input().strip()
    word = step1a(word)
    word = step1b(word)
    word = step1c(word)
    print(word)


10
caresses
caress
ponies
poni
studies
studi
relational
relational
troubling
trouble
filing
file
hopping
hop
sky
sky
cry
cry
running
run
